 Relational schema is like a blueprint of the dataset, it defines things like what tables exist, what variables (columns) each table contains, the data type of each variable, whether missing values are allowed, and how different tables relate to each other. It is useful for social data because social data usually contains many entities (people, events, organizations) spread across multiple datasets. The relational schema records how these datasets connect to each other, which reduces redundancy and improves reproducibility. Primary key is used to uniquely identify each row; foreign key are identifiers for the column, enforces relationships between tables. In the coding lab, each table has its own primary key to uniquely identify each row. The contributions table contains two foreign keys, candidate_id and contributor_id, enabling joins between them. This design reduces duplication because candidate information only needs to be stored once in the candidates table rather than repeating it in every contribution record. When we need combined information, we use a JOIN. For example, joining contributions and candidates on candidate_id = id allows us to retrieve both donation amounts and candidate names in a single query, without storing redundant data.

In [ ]:

import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datetime import date, timedelta

# Load dataset
con = sqlite3.connect("campaign_finance.db")
# Q1 Count the number of rows
print("\n------------------------------")
print("Sanity checks: table sizes")
print("------------------------------")

print(pd.read_sql_query("SELECT COUNT(*) AS n_candidates FROM candidates;", con))
print(pd.read_sql_query("SELECT COUNT(*) AS n_contributors FROM contributors;", con))
print(pd.read_sql_query("SELECT COUNT(*) AS n_contributions FROM contributions;", con))

# Q2 check schema
print(pd.read_sql_query("PRAGMA table_info(candidates)", con))
print(pd.read_sql_query("PRAGMA table_info(contributors)", con))
print(pd.read_sql_query("PRAGMA table_info(contributions)", con))

contributor_id and candidate_id are foreign keys in the contributions table. contributor_id references the primary key of the contributors table, and candidate_id references the primary key of the candidates table. This allows us to join the three tables together — for example, joining contributions and candidates on contributions.candidate_id = candidates.id lets us retrieve both donation amounts and candidate names in a single query.

In [10]:
# join contributions to candidates and compute total contribution by party
query_1 = """
  SELECT
    ca.party,
    SUM(co.amount) AS total_amount,
    COUNT(*) AS num_contributions
  FROM contributions co
  JOIN candidates ca
    ON co.candidate_id = ca.id
  WHERE co.amount > 1000
  GROUP BY ca.party;
"""
party_sum = pd.read_sql_query(query_1, con)
print(party_sum)

# plot
party_sum.plot(kind="bar", x="party", y="total_amount")
plt.title("Total Contributions by Party")
plt.xlabel("Party")
plt.ylabel("Total Amount")
plt.tight_layout()
plt.show()

In [13]:
# check index on contributions
print(pd.read_sql_query("""
    SELECT name, sql
    FROM sqlite_master
    WHERE type='index' AND tbl_name='contributions';
""", con))

# filter by candidate_id
print(pd.read_sql_query("""
    EXPLAIN QUERY PLAN
    SELECT * FROM contributions
    WHERE candidate_id = 1;
""", con))

                         name  \
0  idx_contrib_contributor_id   
1    idx_contrib_candidate_id   
2          idx_contrib_amount   
3            idx_contrib_date   

                                                 sql  
0  CREATE INDEX idx_contrib_contributor_id ON con...  
1  CREATE INDEX idx_contrib_candidate_id   ON con...  
2  CREATE INDEX idx_contrib_amount         ON con...  
3  CREATE INDEX idx_contrib_date           ON con...  
   id  parent  notused                                             detail
0   3       0       61  SEARCH contributions USING INDEX idx_contrib_c...


As can be seen, the contributions table has four indexes: on contributor_id, candidate_id, amount, and date. I filter by candidate_id and the result shows that SQLite uses idx_contrib_candidate_id to search the table, rather than scanning every row. This means SQLite can directly locate matching rows, which is much faster especially when the table is large. Since all commonly filtered columns already have indexes, no additional index is needed for this query.